[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Reading Rows


## What you will be able to do

Read rows back as objects with `select` and `session.exec`, and choose between `all`, `first`, `one`
and `one_or_none` knowing what each does when the number of rows is not what you expected. Write
conditions that combine, order and page results, count rows without loading them, and ask for two
models at once, which gives something other than a list of heroes. Recognize the five failures here:
a `one` that matched twice, a Python `and` between two conditions, a `|` that bound before the
comparisons, a result asked for its length, and a row read as though it were a hero.


## The idea

### The problem

A service reads far more often than it writes. It wants one hero by id, the heroes of one team, the
next twenty in a list, a count for the page numbers, and each hero beside the team it belongs to.
Every one of those is a different shape of answer, and the shape is what makes reading harder than it
looks.

The conditions are the first place Python gets in the way. `Hero.age > 30` is not a comparison Python
evaluates; it builds a piece of SQL. Two of them joined with `and` cannot work, because `and` asks
each side whether it is true and neither knows, and Python's `|` and `&` bind more tightly than `==`,
so the obvious way to write "one or the other" parses as something else entirely. Both fail loudly,
which is the kindest thing they do.

The second place is the answer. `session.exec(select(Hero))` gives heroes, and it is easy to assume
that anything else gives heroes too. Ask for a hero and a team in one query and each result is a row
of two objects, and every line written as though it were a hero fails on the first attribute.

### What select and exec are

> **`select(Hero)`** builds a statement: what to read, from where, under what conditions, in what
> order. It runs nothing. **`session.exec(statement)`** runs it and returns a result to read from:
> **`.all()`** for a list, **`.first()`** for the first row or `None`, **`.one()`** for exactly one,
> which raises when there are none or more than one, and **`.one_or_none()`** for at most one.
> `exec` is SQLModel's: given `select(Hero)` it yields `Hero` objects rather than rows of one hero,
> which is the ordinary case made pleasant. Given **`select(Hero, Team)`** it yields **`Row`**
> objects, which are tuples to unpack.

### Why it works that way

- **A statement is an object, not a string.** `select(Hero).where(...).order_by(...).limit(...)` can
  be built in pieces, passed around and added to, and becomes SQL only when it is run.
- **A condition is built, not evaluated.** `Hero.age > 30` returns a piece of SQL, so Python's `and`,
  `or` and `not` cannot combine them. Use `&`, `|` and `~` with parentheses, or `or_` and `and_`, or
  pass several conditions to `where`, which joins them with `AND`.
- **What comes back follows what you asked for.** One model gives objects, several give rows, and
  columns give values. Nothing changes shape quietly.
- **`col()` is for the type checker, not the database.** `Hero.age` is typed `int | None`, so a
  checker reports `Hero.age > 30` as comparing something that may be `None` with a number.
  `col(Hero.age) > 30` is the same SQL and says that the left side is a column.
  Colab runs no type checker, so nothing here depends on it, and a project with mypy will want it.
- **Counting is a query, not a length.** `len(result.all())` loads every row to count them.
  `select(func.count(Hero.id))` asks the database, which answers with one number.

### Where this shows up

Every list endpoint, every detail page, every report. The paging in the capstone is the body of a
FastAPI route, which the **SQLModel in FastAPI** notebook fills in. The **SQLAlchemy, Deep Dive**
guide's SQL Expressions and Joins and Aggregates notebooks are the long version of the conditions,
the joins and the grouping, and this notebook stays with what SQLModel adds on top.

### What this notebook covers

- `select` and `exec`, and the four ways to take the answer
- Conditions that combine
- Ordering, and a page at a time
- Counting without loading
- Two models in one query, and the rows that come back
- Columns rather than models
- A page of heroes, finished
- Five failures, from a `one` that matched twice to a row read as a hero

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlmodel import Field, Session, SQLModel, create_engine, select


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    age: int | None = None


engine = create_engine("sqlite://")
SQLModel.metadata.create_all(engine)
with Session(engine) as session:
    for name, age in [("Deadpond", None), ("Spider-Boy", 16), ("Rusty-Man", 48)]:
        session.add(Hero(name=name, age=age))
    session.commit()

    statement = select(Hero).where(Hero.age > 20).order_by(Hero.name)
    print("the statement:", " ".join(str(statement).split())[:60], "...")
    for hero in session.exec(statement):
        print("  ", hero.name, hero.age)
```

```
the statement: SELECT hero.id, hero.name, hero.age FROM hero WHERE hero.age ...
   Rusty-Man 48
```

`select(Hero).where(...)` built a statement and read nothing; printing it shows the SQL it will
become. `session.exec` ran it and gave back `Hero` objects, one of them, because only Rusty-Man is
over 20 and the hero with no age is not: a null is not greater than 20, and not less than it either.


## Setup

Ten imports, one of them installed first where it is missing, the cast, two helpers, the classes,
the engine, and the database built and loaded.

- `sqlmodel` is the library, and `SQLModel`, `Field`, `Session`, `create_engine`, `select` and `col`,
  from it, are the classes, the session, the statements and the wrapper that keeps a comparison on an
  optional column readable to a type checker. Colab does not have SQLModel, so the cell installs
  0.0.42 with `pip` where it is missing, and `version` and `PackageNotFoundError`, from
  `importlib.metadata`, `subprocess` and `sys` find out whether it is
- `event`, `insert`, `func` and `or_`, from `sqlalchemy`, are the pragma on every connection, the
  rows loaded without a session, `count` and the other functions the database has, and one of the
  ways to write "either of these"
- `MultipleResultsFound` and `NoResultFound`, from `sqlalchemy.exc`, are what `one()` raises
- `re` takes memory addresses out of a message, `Path` names the database file, and `shutil` removes
  the scratch folder at the start and at the end
- `TEAMS` and `HEROES` are the cast, which `build` loads: eight heroes in three teams, one hero on no
  team and one team with no heroes, which the joins below depend on

`hero_engine` and `build` are the **Engine and create_all** and **Sessions** notebooks' engine and
loader, and the two classes are the ones **Engine and create_all** wrote.


In [1]:
import re
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from sqlalchemy import event, func, insert, or_
from sqlalchemy.exc import MultipleResultsFound, NoResultFound
from sqlmodel import Field, Session, SQLModel, col, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes in",
          len(session.exec(select(Team)).all()), "teams")


sqlmodel 0.0.42 | 8 heroes in 3 teams


## Worked examples

### select and exec, and the four ways to take the answer

`select(Hero)` is every hero. What comes back depends on which method the result is asked for:


In [2]:
with Session(engine) as session:
    everyone = session.exec(select(Hero)).all()
    print("all()        :", len(everyone), "heroes, the first is", everyone[0].name)
    print("first()      :", session.exec(select(Hero).where(Hero.age == 48)).first().name)
    print("one()        :", session.exec(select(Hero).where(Hero.name == "Deadpond")).one().name)
    print("one_or_none():", session.exec(select(Hero).where(Hero.name == "Nobody")).one_or_none())


all()        : 8 heroes, the first is Deadpond
first()      : Rusty-Man
one()        : Deadpond
one_or_none(): None


Four methods, four contracts. `all()` gives a list, `first()` gives the first row or `None` and
fetches no more than one, `one()` insists there is exactly one and raises otherwise, and
`one_or_none()` allows none but not two. `one()` is the right choice when a row must exist, because
the alternative is an `AttributeError` on `None` somewhere later, and the first two of the Common
errors are what it raises.

| Method | No rows | One row | Several rows |
|---|---|---|---|
| `all()` | `[]` | a list of one | a list |
| `first()` | `None` | the row | the first row |
| `one()` | raises `NoResultFound` | the row | raises `MultipleResultsFound` |
| `one_or_none()` | `None` | the row | raises `MultipleResultsFound` |

### Conditions that combine

Several conditions can be passed to one `where`, or chained, or joined with `&` and `|` in
parentheses:


In [3]:
with Session(engine) as session:
    grown = select(Hero).where(Hero.age > 30, Hero.age < 50)        # two conditions, joined with AND
    print("30 to 50   :", [hero.name for hero in session.exec(grown)])

    chained = select(Hero).where(Hero.age > 30).where(Hero.team_id == 1)
    print("and on team 1:", [hero.name for hero in session.exec(chained)])

    either = select(Hero).where((Hero.name == "Deadpond") | (Hero.name == "Rusty-Man"))
    print("either name:", [hero.name for hero in session.exec(either)])

    spelled = select(Hero).where(or_(Hero.team_id == 2, col(Hero.team_id).is_(None)))
    print("Z-Force or no team:", [hero.name for hero in session.exec(spelled)])


30 to 50   : ['Tarantula', 'Black Lion', 'Dr. Weird', 'Rusty-Man']
and on team 1: ['Tarantula', 'Rusty-Man', 'Captain North America']
either name: ['Deadpond', 'Rusty-Man']
Z-Force or no team: ['Deadpond', 'Black Lion', 'Dr. Weird', 'Princess Sure-E']


`where(a, b)` and `.where(a).where(b)` are the same thing, both `AND`. `|` is `OR` and `&` is `AND`,
and both need parentheses around each comparison, because `|` binds more tightly than `==` and the
third of the Common errors is what happens without them. `or_(...)` says the same thing without
punctuation, which is easier to read when there are three or four.

The last line has two other things worth naming. A null is compared with `is_(None)`, not `== None`,
because SQL has no equality with null. And `col(...)` is the wrapper The idea section mentioned: it
changes no SQL, and it is what keeps a type checker quiet about a column typed `int | None`.

### Ordering, and a page at a time

`order_by`, `offset` and `limit` are the body of every list that has more rows than fit on a screen:


In [4]:
with Session(engine) as session:
    for page in (0, 1, 2):
        statement = select(Hero).order_by(Hero.name).offset(page * 3).limit(3)
        print(f"page {page + 1}:", [hero.name for hero in session.exec(statement)])

    oldest = select(Hero).order_by(col(Hero.age).desc()).limit(3)
    print("oldest three:", [(hero.name, hero.age) for hero in session.exec(oldest)])


page 1: ['Black Lion', 'Captain North America', 'Deadpond']
page 2: ['Dr. Weird', 'Princess Sure-E', 'Rusty-Man']
page 3: ['Spider-Boy', 'Tarantula']
oldest three: [('Captain North America', 93), ('Rusty-Man', 48), ('Dr. Weird', 36)]


Three pages of three, and the last has two. `order_by` is not optional for paging: without it a
database may return rows in any order it likes, and page 2 can repeat a hero from page 1 or miss one
entirely. `col(Hero.age).desc()` orders the other way, and the three oldest come back. Where the
heroes with no age belong in that order is the database's decision, and databases disagree about it,
so a query whose answer depends on it says `.nulls_last()` or `.nulls_first()` and leaves nothing to
the default.

### Counting without loading

The number of heroes is a question for the database, not a length in Python:


In [5]:
with Session(engine) as session:
    print("count      :", session.exec(select(func.count(Hero.id))).one())
    print("on team 1  :", session.exec(select(func.count(Hero.id)).where(Hero.team_id == 1)).one())
    print("oldest age :", session.exec(select(func.max(Hero.age))).one())
    print("loaded     :", len(session.exec(select(Hero)).all()))


count      : 8
on team 1  : 4
oldest age : 93
loaded     : 8


The first three read one row each, whatever the size of the table. The last loaded every hero into
memory to ask how many there were, which is the same answer and a different cost, and it is what a
list endpoint must not do to produce a total. A result object has no `count` on it either, which is
the fourth of the Common errors.

### Two models in one query, and the rows that come back

A hero and a team in one statement is a join, and the answer is no longer a list of heroes:


In [6]:
with Session(engine) as session:
    rows = session.exec(select(Hero, Team).join(Team)).all()
    print("what came back:", type(rows[0]).__name__, "| its parts:", rows[0]._fields)
    for hero, team in rows[:3]:
        print(f"  {hero.name:<22} {team.name}")
    print("heroes with a team:", len(rows), "of", len(HEROES))


what came back: Row | its parts: ('Hero', 'Team')
  Deadpond               Z-Force
  Spider-Boy             Preventers
  Rusty-Man              Preventers
heroes with a team: 7 of 8


Each result is a `Row`, a tuple of the two objects that were asked for, and unpacking it with
`for hero, team in ...` is how it is read. Writing `row.name` instead is the fifth of the Common
errors.

Seven of the eight heroes came back, because a join keeps only the rows that match, and Princess
Sure-E is on no team. `isouter=True` keeps that hero too, with nothing where the team would be:


In [7]:
with Session(engine) as session:
    rows = session.exec(select(Hero, Team).join(Team, isouter=True)).all()
    for hero, team in rows:
        if team is None or hero.name == "Deadpond":
            print(f"  {hero.name:<22} {team.name if team else '(no team)'}")
    print("rows:", len(rows))

    teamless = select(Team, Hero).join(Hero, isouter=True).where(col(Hero.id).is_(None))
    print("teams with nobody in them:", [team.name for team, hero in session.exec(teamless)])


  Deadpond               Z-Force
  Princess Sure-E        (no team)
rows: 8
teams with nobody in them: ['Wakaland Guard']


The outer join keeps every hero and puts `None` where a team would be, which is the whole of the
difference. Turned around, it also answers the other question a list page asks: which teams have
nobody in them, found by keeping every team and looking for the rows where no hero matched.

### Columns rather than models

A statement can ask for columns instead. One column gives plain values, and several give rows again:


In [8]:
with Session(engine) as session:
    names = session.exec(select(Hero.name).where(Hero.age > 30)).all()
    print("one column  :", names, type(names[0]).__name__)

    pairs = session.exec(select(Hero.name, Team.name).join(Team).order_by(Hero.name).limit(3)).all()
    print("two columns :", pairs, type(pairs[0]).__name__)
    print("unpacked    :", [f"{hero} of the {team}" for hero, team in pairs])


one column  : ['Tarantula', 'Black Lion', 'Dr. Weird', 'Rusty-Man', 'Captain North America'] str
two columns : [('Black Lion', 'Z-Force'), ('Captain North America', 'Preventers'), ('Deadpond', 'Z-Force')] Row
unpacked    : ['Black Lion of the Z-Force', 'Captain North America of the Preventers', 'Deadpond of the Z-Force']


Asking for one column gives the values themselves, which is what a list of names or ids wants and is
far less to carry than whole objects. Asking for two gives rows again, unpacked the same way. Neither
gives objects the session is watching, so nothing read this way can be changed and committed; for
that, read the model.

### A page of heroes, finished

The pieces of this notebook in one function. `page_of_heroes` is the body of a list endpoint: a
filter, an order, a page, and a total that does not load the table:


In [9]:
def page_of_heroes(session, team=None, page=1, per_page=3):
    """One page of heroes with their teams, and how many there are in total."""
    listed = select(Hero, Team).join(Team, isouter=True)
    counted = select(func.count(Hero.id))
    if team is not None:
        listed = listed.where(Team.name == team)
        counted = counted.join(Team).where(Team.name == team)
    rows = session.exec(listed.order_by(Hero.name).offset((page - 1) * per_page).limit(per_page)).all()
    return {"total": session.exec(counted).one(),
            "page": page,
            "heroes": [{"name": hero.name, "team": team.name if team else None} for hero, team in rows]}


with Session(engine) as session:
    for asked in ({}, {"page": 3}, {"team": "Preventers"}, {"team": "Wakaland Guard"}):
        print(asked, "->", page_of_heroes(session, **asked))


{} -> {'total': 8, 'page': 1, 'heroes': [{'name': 'Black Lion', 'team': 'Z-Force'}, {'name': 'Captain North America', 'team': 'Preventers'}, {'name': 'Deadpond', 'team': 'Z-Force'}]}
{'page': 3} -> {'total': 8, 'page': 3, 'heroes': [{'name': 'Spider-Boy', 'team': 'Preventers'}, {'name': 'Tarantula', 'team': 'Preventers'}]}
{'team': 'Preventers'} -> {'total': 4, 'page': 1, 'heroes': [{'name': 'Captain North America', 'team': 'Preventers'}, {'name': 'Rusty-Man', 'team': 'Preventers'}, {'name': 'Spider-Boy', 'team': 'Preventers'}]}
{'team': 'Wakaland Guard'} -> {'total': 0, 'page': 1, 'heroes': []}


Four calls, four answers. The first is page one of everyone; the third page is the last two, and the
total stays 8 whichever page is asked for, because the count is its own query with no `limit` on it.
The Preventers have four, and Wakaland Guard has none, which is a total of 0 and an empty list rather
than an error.

### Where each part came from

| In `page_of_heroes` | What it relies on | The section that showed it |
|---|---|---|
| `select(Hero, Team).join(Team, isouter=True)` | two models in one query, and the heroes with no team kept | Two models in one query |
| `listed.where(Team.name == team)` | a condition added to a statement already built | Conditions that combine |
| `.order_by(...).offset(...).limit(...)` | a page in a fixed order | Ordering, and a page at a time |
| `select(func.count(Hero.id))` | a total the database works out | Counting without loading |
| `for hero, team in rows` | a `Row` unpacked into the two objects asked for | Two models in one query |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/05-reading-rows-solutions.ipynb).

**1.** Print the names of every hero whose age is known, oldest first.


In [10]:
# your code here


**2.** Find the hero whose secret name is `Tommy Sharp` with `one()`, and print what
`one_or_none()` gives for a secret name nobody has.


In [11]:
# your code here


**3.** Print the heroes who are either on the Z-Force or have no team at all, in two ways: with `|`
and parentheses, and with `or_`.


In [12]:
# your code here


**4.** Print how many heroes each team has, without loading the heroes, using `func.count` and
`group_by`. Keep the team that has none.


In [13]:
# your code here


**5.** Print every hero beside the headquarters of the team they belong to, as `name: headquarters`,
with the heroes on no team left out.


In [14]:
# your code here


**6.** Write `heroes_named(session, fragment)`, which returns the names of the heroes whose name
contains a fragment, in alphabetical order. `col(Hero.name).contains(...)` is the condition.


In [15]:
# your code here


## Common errors

### sqlalchemy.exc.MultipleResultsFound: Multiple rows were found when exactly one was required


In [16]:
with Session(engine) as session:
    session.exec(select(Hero).where(Hero.team_id == 1)).one()


MultipleResultsFound: Multiple rows were found when exactly one was required

Four heroes are on team 1, and `one()` promised there would be exactly one. The other half of the
same promise is `NoResultFound`, which is what a condition that matches nothing raises:


In [17]:
with Session(engine) as session:
    try:
        session.exec(select(Hero).where(Hero.name == "Nobody")).one()
    except NoResultFound as error:
        print(type(error).__name__ + ":", error)

    print("first()      :", session.exec(select(Hero).where(Hero.team_id == 1)).first().name)
    print("one_or_none():", session.exec(select(Hero).where(Hero.name == "Nobody")).one_or_none())


NoResultFound: No row was found when one was required
first()      : Spider-Boy
one_or_none(): None


Which one to reach for is a question about the data, not about style. `one()` where the row must
exist and a missing one is a bug; `one_or_none()` where it may be missing and the caller will decide,
which is what a route that answers 404 needs; `first()` where several may match and any will do.

### TypeError: Boolean value of this clause is not defined


In [18]:
select(Hero).where(Hero.age > 30 and Hero.age < 50)


TypeError: Boolean value of this clause is not defined

Python's `and` asks its left side whether it is true, and `Hero.age > 30` is a piece of SQL that has
no answer to that: it is neither true nor false until a database runs it against a row. Rather than
guess, SQLAlchemy raises.

Write it as two conditions, or with `&` and parentheses:


In [19]:
with Session(engine) as session:
    both = select(Hero).where(Hero.age > 30, Hero.age < 50)
    print("two conditions:", [hero.name for hero in session.exec(both)])
    print("with &        :", [hero.name for hero in
                              session.exec(select(Hero).where((Hero.age > 30) & (Hero.age < 50)))])


two conditions: ['Tarantula', 'Black Lion', 'Dr. Weird', 'Rusty-Man']
with &        : ['Tarantula', 'Black Lion', 'Dr. Weird', 'Rusty-Man']


### TypeError: unsupported operand type(s) for |: 'str' and 'InstrumentedAttribute'


In [20]:
select(Hero).where(Hero.name == "Deadpond" | Hero.name == "Rusty-Man")


TypeError: unsupported operand type(s) for |: 'str' and 'InstrumentedAttribute'

The message names the two things Python actually tried to combine: the string `"Deadpond"` and the
column `Hero.name`. `|` binds more tightly than `==`, so that line means
`Hero.name == ("Deadpond" | Hero.name) == "Rusty-Man"`, which is not what anybody meant. The same is
true of `&`.

Parentheses around each comparison fix it, and `or_` avoids the question:


In [21]:
with Session(engine) as session:
    print("parentheses:", [hero.name for hero in
                           session.exec(select(Hero).where((Hero.name == "Deadpond") | (Hero.name == "Rusty-Man")))])
    print("or_        :", [hero.name for hero in
                           session.exec(select(Hero).where(or_(Hero.name == "Deadpond", Hero.name == "Rusty-Man")))])


parentheses: ['Deadpond', 'Rusty-Man']
or_        : ['Deadpond', 'Rusty-Man']


### AttributeError: 'ScalarResult' object has no attribute 'count'


In [22]:
with Session(engine) as session:
    print(session.exec(select(Hero)).count)


AttributeError: 'ScalarResult' object has no attribute 'count'

What `exec` returns is a result to read rows from, not a collection. It has `all`, `first`, `one` and
`one_or_none`, and it can be iterated once, and that is all: there is no length, no indexing and no
`count`, because the rows have not been fetched and counting them would mean fetching them.

Ask the database for the number instead, or, when the rows are wanted anyway, take the length of the
list:


In [23]:
with Session(engine) as session:
    print("from the database:", session.exec(select(func.count(Hero.id))).one())
    heroes = session.exec(select(Hero)).all()
    print("from the list    :", len(heroes))


from the database: 8
from the list    : 8


### AttributeError: name


In [24]:
with Session(engine) as session:
    for hero in session.exec(select(Hero, Team).join(Team)):
        print(hero.name)


AttributeError: name

The variable is called `hero` and holds a `Row`, a tuple of the hero and the team that were asked
for. A `Row` has the names of the things in it, `Hero` and `Team`, and not their attributes, so
`name` is not one of its fields and the message is that bare.

Unpack the row, which is what the two-name `for` does, or name the row for what it is:


In [25]:
with Session(engine) as session:
    for hero, team in session.exec(select(Hero, Team).join(Team).limit(2)):
        print(f"  {hero.name} of the {team.name}")

    for row in session.exec(select(Hero, Team).join(Team).limit(2)):
        print(f"  {row.Hero.name} of the {row.Team.name}")


  Deadpond of the Z-Force
  Spider-Boy of the Preventers
  Deadpond of the Z-Force
  Spider-Boy of the Preventers


Last, the engine lets go of the file, and this cell removes the scratch folder with the database in
it:


In [26]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `select(Hero)` builds a statement and runs nothing; `session.exec` runs it and gives `Hero`
  objects, and `all`, `first`, `one` and `one_or_none` differ in what they do when the count is not
  what you expected.
- Conditions are built, not evaluated: use several arguments to `where`, or `&`, `|` and `~` with
  parentheses, or `and_` and `or_`, and `is_(None)` for a null.
- `order_by` is what makes paging repeatable, and `offset` and `limit` are the page.
- `select(func.count(Hero.id))` counts in the database; a result object has no `count` and no length.
- `select(Hero, Team)` gives `Row` objects to unpack, `isouter=True` keeps the rows with nothing on
  the other side, and `select(Hero.name)` gives plain values.


## What is next

The **Validation and table=True** notebook is about the check that does not happen: a table model's
constructor takes whatever it is given, a validator written on the class never runs, and the value
that was never checked is found much later, by a column or by the model that finally validates it.


---

&#8592; **Previous:** [Field Types and Defaults](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/04-field-types-and-defaults.ipynb)  &nbsp;·&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
